# Project :- Movie-Recommendation-System

## Phase 1 — Notebook Lifecycle Setup, Library Imports & Raw Data Loading

In [1]:
# Importing Core Data Libraries
import pandas as pd
import numpy as np
import os
import ast

print("Core libraries imported successfully!")


Core libraries imported successfully!


In [3]:
# Defining the Exact Raw Data Path

DATA_PATH = r"C:\Users\lenovo\OneDrive\Desktop\movie-recommendation-system\data\raw\tmdb_5000_credits.csv"

if os.path.exists(DATA_PATH):
    print("RIGHT PATH!!! FILE EXIST")
else:
    print("FILE DOES NOT EXIST")


RIGHT PATH!!! FILE EXIST


In [5]:
# Reading the CSV File into a Pandas DataFrame
df = pd.read_csv(DATA_PATH)

print("Data load SUCCESSFULLY")
print(f"Total Movies INFO data: {df.shape[0]} rows aur {df.shape[1]} columns.")


Data load SUCCESSFULLY
Total Movies INFO data: 4803 rows aur 4 columns.


In [6]:
# Inspecting columns and first 5 rows of data
df.head(5)


,movie_id,title,cast,crew
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de..."
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de..."


In [7]:
# Saving Today's Progress into the 'processed' folder
processed_dir = r"C:\Users\lenovo\OneDrive\Desktop\movie-recommendation-system\data\processed"

# Agar folder accidentally delete ho gaya ho toh auto-create ho jayega
os.makedirs(processed_dir, exist_ok=True)

# Aaj ke loaded data ko processed folder me save kar rahe hain
df.to_csv(os.path.join(processed_dir, "day1_loaded_data.csv"), index=False)
print("Day 1 work successfully saved in 'data/processed' folder")


Day 1 work successfully saved in 'data/processed' folder


# Project :- Movie-Recommendation-System

## Phase 2 : Data Cleaning & Feature Extraction

In [8]:
import pandas as pd
import numpy as np
import ast
import os

processed_path = "../data/processed/day1_loaded_data.csv"
df = pd.read_csv(processed_path)


In [9]:
def convert_to_list(obj):
    try:
        return ast.literal_eval(obj)
    except (ValueError, SyntaxError):
        return []

df['cast'] = df['cast'].apply(convert_to_list)
df['crew'] = df['crew'].apply(convert_to_list)


In [10]:
def extract_top_actors(obj):
    actors_list = []
    counter = 0
    for i in obj:
        if counter < 5:
            clean_name = i['name'].replace(" ", "")
            actors_list.append(clean_name)
            counter += 1
        else:
            break
    return actors_list

df['actors'] = df['cast'].apply(extract_top_actors)


In [11]:
def fetch_director_name(obj):
    for i in obj:
        if i['job'] == 'Director':
            return [i['name'].replace(" ", "")]
    return []

df['director'] = df['crew'].apply(fetch_director_name)


In [12]:
df[['title', 'actors', 'director']].head(5)

,title,actors,director
0,Avatar,"[SamWorthington, ZoeSaldana, SigourneyWeaver, ...",[JamesCameron]
1,Pirates of the Caribbean: At World's End,"[JohnnyDepp, OrlandoBloom, KeiraKnightley, Ste...",[GoreVerbinski]
2,Spectre,"[DanielCraig, ChristophWaltz, LéaSeydoux, Ralp...",[SamMendes]
3,The Dark Knight Rises,"[ChristianBale, MichaelCaine, GaryOldman, Anne...",[ChristopherNolan]
4,John Carter,"[TaylorKitsch, LynnCollins, SamanthaMorton, Wi...",[AndrewStanton]


In [13]:
# final save
processed_dir = "../data/processed"
df.to_csv(os.path.join(processed_dir, "day2_parsed_data.csv"), index=False)

# Project :- Movie-Recommendation-System

## Phase 3 : Text Preprocessing, Stemming, aur Feature Vectorization

In [14]:
# Load parsed data from Day 2
import pandas as pd
import numpy as np
import os
from sklearn.feature_extraction.text import CountVectorizer

processed_path = "../data/processed/day2_parsed_data.csv"
df = pd.read_csv(processed_path)

print("Day 2 parsed data loaded successfully!")
print(f"Current Shape: {df.shape}")


Day 2 parsed data loaded successfully!
Current Shape: (4803, 6)


In [15]:
# Dropping duplicate movies and missing titles
df.drop_duplicates(subset=['movie_id'], inplace=True)
df.dropna(subset=['title'], inplace=True)

print(f"Data validated! Shape after basic structural safety checks: {df.shape}")


Data validated! Shape after basic structural safety checks: (4803, 6)


In [16]:
# Converting title and keywords into word lists
# Note: cast and crew lists are already handled, strings need proper splitting if loaded as string
df['title_list'] = df['title'].apply(lambda x: [x.replace(" ", "").lower()])

# Safely string to list conversion for engineered fields if loaded as raw string from CSV
def string_to_list(text_data):
    if pd.isna(text_data):
        return []
    if isinstance(text_data, str) and text_data.startswith('['):
        try:
            import ast
            return ast.literal_eval(text_data)
        except:
            return text_data.split()
    return str(text_data).split()

df['actors'] = df['actors'].apply(string_to_list)
df['director'] = df['director'].apply(string_to_list)

print("Structural features properly formatted into list tokens!")


Structural features properly formatted into list tokens!


In [17]:
# Feature Engineering — Creating the Final 'Tags' Column
# Merging all structural properties into a single column
df['tags'] = df['title_list'] + df['actors'] + df['director']

# List format ko wapas ek space-separated text paragraph me convert 
df['tags'] = df['tags'].apply(lambda x: " ".join(x))

print("Tags matrix paragraph column constructed!")

# Preview checking
print(df[['title', 'tags']].head(5))


Tags matrix paragraph column constructed!
                                      title  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   
2                                   Spectre   
3                     The Dark Knight Rises   
4                               John Carter   

                                                tags  
0  avatar SamWorthington ZoeSaldana SigourneyWeav...  
1  piratesofthecaribbean:atworld'send JohnnyDepp ...  
2  spectre DanielCraig ChristophWaltz LéaSeydoux ...  
3  thedarkknightrises ChristianBale MichaelCaine ...  
4  johncarter TaylorKitsch LynnCollins SamanthaMo...  


In [19]:
!py -m pip install nltk


   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/1.8 MB ? eta -:--:--
   ----------------- ---------------------- 0.8/1.8 MB 2.3 MB/s eta 0:00:01
   ----------------------------- ---------- 1.3/1.8 MB 2.3 MB/s eta 0:00:01
   ---------------------------------- ----- 1.6/1.8 MB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 2.3 MB/s  0:00:00


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [20]:
import nltk

nltk.download('punkt')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


True

In [21]:
# Applying Porter Stemmer to normalize words
import nltk
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

def stem_text(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)

df['tags'] = df['tags'].apply(stem_text)
print("Stemming process completed successfully!")


Stemming process completed successfully!


In [22]:
# Feature Vectorization — Bags of Words Matrix Construction
# Cell 6: Vectorizing tags into numeric arrays
cv = CountVectorizer(max_features=5000, stop_words='english')

# Transforming string paragraphs into a massive NumPy multidimensional array
vectors = cv.fit_transform(df['tags']).toarray()

print(f"ML Processing Complete! Matrix Vectors Dimension Shape: {vectors.shape}")


ML Processing Complete! Matrix Vectors Dimension Shape: (4803, 5000)


In [23]:
# Exporting to Cleaned Directory
# Dumping processed output to data/cleaned folder
cleaned_dir = "../data/cleaned"
os.makedirs(cleaned_dir, exist_ok=True)

# Main dataframe save kar rahe hain jisme final text tags hain
df.to_csv(os.path.join(cleaned_dir, "day3_cleaned_tags.csv"), index=False)

# NumPy vectors matrix ko alag binary checkpoint me lock kar rahe hain
np.save(os.path.join(cleaned_dir, "day3_vectors.npy"), vectors)

print("Day 3 work successfully archived inside data/cleaned/ directory!")


Day 3 work successfully archived inside data/cleaned/ directory!


# Project :- Movie-Recommendation-System¶

## Phase 4 :  Cosine Similarity Modeling aur Serialization

### Cleaned Tags & Vectors Matrix

In [1]:
# Load cleaned text and generated arrays matrix
import pandas as pd
import numpy as np
import os
import pickle
from sklearn.metrics.pairwise import cosine_similarity

cleaned_dir = "../data/cleaned"

# Loading DataFrame and NumPy binary arrays
df = pd.read_csv(os.path.join(cleaned_dir, "day3_cleaned_tags.csv"))
vectors = np.load(os.path.join(cleaned_dir, "day3_vectors.npy"))

print("Day 3 processed outputs successfully re-loaded into memory!")
print(f"Movies Table Shape: {df.shape} | Numerical Matrix Shape: {vectors.shape}")


Day 3 processed outputs successfully re-loaded into memory!
Movies Table Shape: (4803, 8) | Numerical Matrix Shape: (4803, 5000)


In [2]:
#  Calculating Angle Proximity between 4803 movies vectors
similarity = cosine_similarity(vectors)

print(f"Similarity Engine Created! Matrix Dimensions Shape: {similarity.shape}")


Similarity Engine Created! Matrix Dimensions Shape: (4803, 4803)


In [3]:
# Core recommendation sorting logic algorithm

def recommend_movie(movie_name):

    try:
        movie_index = df[df['title'].str.lower() == movie_name.lower()].index[0]
    except IndexError:
        print(f"sorry!!! '{movie_name}' movie is not available in database")
        return
    
    distances = similarity[movie_index]
    
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]

    print(f"--- Top 5 Recommended Movies for '{movie_name}' ---")
    for i in movies_list:
        print(df.iloc[i[0]]['title'])

# Testing the model workflow locally
recommend_movie('Avatar')


--- Top 5 Recommended Movies for 'Avatar' ---
Aliens
Backmask
Guess Who
Drumline
The Cold Light of Day


In [5]:
# Exporting Artifacts for Web Deployment
# Cell 4: Creating pickling binary structures into artifacts folder
artifacts_dir = "../artifacts"
os.makedirs(artifacts_dir, exist_ok=True)

# 1. Dataframe ko directly dict me badalkar save karenge
# Kyunki different pandas versions pipelines break kar sakte hain web cloud par
pickle.dump(df.to_dict(), open(os.path.join(artifacts_dir, "movie_dict.pkl"), "wb"))

# 2. Calculated massive similarity square arrays lock karein
pickle.dump(similarity, open(os.path.join(artifacts_dir, "similarity.pkl"), "wb"))

print("Model Architecture binaries serialized successfully inside artifacts/ layout!")


Model Architecture binaries serialized successfully inside artifacts/ layout!
